### RAG Pipeline - Data INgestion to VEctor DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\abhin\AppData\Local\Temp\ipykernel_29200\2821909192.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader
c:\Users\abhin\Dropbox\PC\Downloads\projects\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory."""

    all_documents = []
    pdf_dir = Path(pdf_directory)
    for pdf_file in pdf_dir.glob("**/*.pdf"):
        print(f"Processing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = str(pdf_file)
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Processed {len(documents)} documents from {pdf_file}.")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    print(f"Total documents processed: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/PDF")

Processing ..\data\PDF\2607.08052v1.pdf...


Processed 45 documents from ..\data\PDF\2607.08052v1.pdf.
Total documents processed: 45


In [3]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"First chunk: {split_docs[0].page_content[:100]}...")
        print(f"Metadata of first chunk: {split_docs[0].metadata}")
        print(f"Last chunk: {split_docs[-1].page_content[:100]}...")

    return split_docs


In [4]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Splitted 45 documents into 127 chunks.
First chunk: COMPLEX CURVES IN O-MINIMAL GEOMETRY
SPENCER DEMBNER
Abstract. There has recently been considerable ...
Metadata of first chunk: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:8def8d8)', 'creationdate': '', 'source': '..\\data\\PDF\\2607.08052v1.pdf', 'file_path': '..\\data\\PDF\\2607.08052v1.pdf', 'total_pages': 45, 'format': 'PDF 1.7', 'title': 'Complex curves in o-minimal geometry', 'author': 'Spencer Dembner', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': '..\\data\\PDF\\2607.08052v1.pdf', 'file_type': 'pdf'}
Last chunk: Transcendence properties of the exponential function in several
variables. Springer-Verlag, Berlin, ...


In [5]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimenesion: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        Args:
            texts: List of text strings
        Returns:
            np.ndarray: Array of embeddings
        """
        if not self.model:
            raise ValueError("Embedding model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3135.05it/s]


Model loaded successfully. Embedding dimenesion: 384


### Vector Store using ChromaDB

In [16]:
class VectorStore:
    """ Manages document embeddings in ChromaDB vector store """

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the ChromaDB data
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initialize ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}")
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
            metadata={"description": "Collection of PDF document embeddings"})
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.
        
        Args:
            documents: List of Langchain Documents
            embeddings: Corresponding embeddings of the documents as a numpy array
        """

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store.")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{i}_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            print("=" * 80)
            print("PAGE CONTENT")
            print(doc.page_content[:1000])   # print first 1000 characters
            print("=" * 80)

            #Document content
            documents_texts.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

        # Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error occurred while adding documents to the vector store: {e}")

In [17]:
vector_store = VectorStore()
vector_store

Initializing ChromaDB client with persist directory: ../data/vector_store
ChromaDB collection 'pdf_documents' initialized successfully.


In [18]:
### Convert text to embeddings
texts = [doc.page_content for doc in chunks]
texts

###Generate Embeddings
embeddings = embedding_manager.generate_embeddings(texts)
embeddings

### Store in Vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 127 texts


Batches: 100%|██████████| 4/4 [00:05<00:00,  1.44s/it]


Generated embeddings with shape: (127, 384)
Adding 127 documents to the vector store.
PAGE CONTENT
COMPLEX CURVES IN O-MINIMAL GEOMETRY
SPENCER DEMBNER
Abstract. There has recently been considerable progress relating o-
minimality to complex analytic geometry. Yet almost nothing is known
about coherent cohomology or the classification of vector bundles, even
for curves. In Ran and similar structures, we show that cohomology of
noncompact curves is concentrated entirely at punctures. As an applica-
tion, we compute the cohomology of the structure sheaf on the affine line
and describe a connection to Diophantine approximation. Finally, we use
similar techniques to characterize which definable Riemann surfaces have
definable compactifications. The proofs are based on a careful analysis
of boundary behavior for definable holomorphic functions.
1. Introduction
Let S be an o-minimal structure on R. Peterzil and Starchenko [PS10]
developed a theory of definable complex analysis, by replacing 

In [10]:
# Check vector store collection
print(f"Collection name: {vector_store.collection_name}")
print(f"Total documents in collection: {vector_store.collection.count()}")

# Get sample documents
results = vector_store.collection.get(limit=3)
print(f"\nSample documents:")
for i, doc_id in enumerate(results['ids'][:3]):
    print(f"  - {doc_id}")

Collection name: pdf_documents
Total documents in collection: 1092

Sample documents:
  - doc_0_1a97be3d_0
  - doc_1_5e11b3fc_1
  - doc_2_bf396c64_2


### Retriever Pipeline from Vector Store

In [11]:
class RAGRetriever:
    """ Retrieves relevant documents based on user queries
      """
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAGRetriever.
        Args:
            vector_store: Instance of VectorStore
            embedding_manager: Instance of EmbeddingManager
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -1.0) -> List[Dict[str, Any]]:
        """ Retrieve relevant documents based on the query.
        
        Args:
            query: User query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score to consider a document relevant
            
            Returns:
                List of dictionaries containing document metadata and content
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            
            # print(vector_store.collection.get())
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, documents, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    # print(doc_id, distance, similarity_score)
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": documents,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}.")

            else:
                print("No documents retrieved from the vector store.")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)


In [12]:
rag_retriever.retrieve("Who is Abhinav Singh?")

Retrieving documents for query: 'Who is Abhinav Singh?'
Top K: 5, Score Threshold: -1.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.87it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents above the score threshold of -1.0.


[{'id': 'doc_4_4e5873bd_4',
  'content': 'FORM\nITR-2\nINDIAN INCOME TAX RETURN\n[For Individuals and HUFs not having income from profits and gains of business or profession]\n(Please see Rule 12 of the Income-tax Rules, 1962)\n(Please refer instructions)\nAssessment Year\n2025-26\nPART A-GENERAL\nPERSONAL INFORMATION\n(A1) First Name\nABHINAV\n(A2) Middle Name\n(A3) Last Name\nSINGH\n(A4) PAN\nKIWPS5719E\n(A5) Status\nIndividual\n(A6) Flat/Door/Block No.\nFlat No-404 Plot No-34 Aman Chain Puri Society\n(A7) Name of Premises/Building/Village\nSector-2 Ballabgarh\n(A8) Road/Street/Post Office\nBallabgarh S.O\n(A9) Area/locality\nBallabgarh\n(A10) Town/City/District\nFARIDABAD\n(A11) State\n12-Haryana\n(A12) Country/Region\n91-India\n(A13) Pin Code/Zip Code\n121004\n(A16) Residential/Office Phone Number with\nSTD/ISD code\nMobile No. 1\n91 9506084155\n(A17) Mobile No. 2\n(A18) Email Address-1 (self)\nabhinavsingh649@gmail.com\n(A19) Email Address-2\n(A14) Date of Birth/Formation (DD/MM/Y

### Integrating VercotDB Contect Pipeline with LLM Output

In [24]:
### Simple Pipeline for RAG with Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv

### Initialize Gemini LLM
gemini240598_api_key = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    google_api_key=gemini240598_api_key,
    model="gemini-3.5-flash",
    temperature=0.7,
    max_tokens=1024
)

### Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k = 3):
    """ Simple RAG function: retrieve context + generate response
    Args:
        query: User query string
        retriever: Instance of RAGRetriever
        llm: Instance of LLM (e.g., ChatGoogleGenerativeAI)
        top_k: Number of top documents to retrieve
    Returns:
        Generated response from the LLM
    """
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    
    if not retrieved_docs:
        return "No relevant documents found."

    # Combine retrieved documents into a single context string
    # Build context using only the document text
    context="\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""
    Use the following context to answer the question.

    Context:
    {context}

    Question:
    {query}

    Rules:
    # - Answer only from the provided context.
    # - Return only the answer.
    # - Do not include any metadata, document numbers, signatures, hashes, IDs, or extra fields.
    - If the answer is not present in the context, respond exactly with:
    Answer not found in the provided documents.
    """
    
    response = llm.invoke(prompt)
    if isinstance(response.content, list):
        answer = "".join(item.get("text", "") for item in response.content if isinstance(item, dict))
    else:
        answer = str(response.content)

    return answer


In [26]:
answers = rag_simple("Summarize the document", rag_retriever, llm, top_k=3)
print(answers)

Retrieving documents for query: 'Summarize the document'
Top K: 3, Score Threshold: -1.0
Generating embeddings for 1 texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the score threshold of -1.0.
The document consists of repeated tables detailing the utilisation of Alternate Minimum Tax (AMT) credit for Assessment Years 2013-14 to 2024-25. For all listed assessment years, all values—including Gross AMT Credit, Set-off in earlier assessment years, Balance brought forward to the current assessment year, AMT Credit Utilised during the Current Assessment Year, and Balance AMT Credit Carried Forward—are recorded as 0. Consequently, the total tax credit under section 115JD utilised during the year is 0, and the total AMT liability available for credit in subsequent assessment years is also 0.


### Enhanced RAG Pieline Features

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=-1.0, return_context=False):
    """Advanced RAG pipeline with extra features."""
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not retrieved_docs:
        return {
            'answers': 'No relevant documents found.',
            'sources': [],
            'confidence_score': 0.0,
            'context': ''
        }

    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ''
    if not context:
        return {
            'answers': 'No relevant context found to answer the question.',
            'sources': [],
            'confidence_score': 0.0,
            'context': ''
        }

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in retrieved_docs]

    confidence = max([doc['similarity_score'] for doc in retrieved_docs], default=0.0)

    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)

    if hasattr(response, 'content'):
        content = response.content
        if isinstance(content, list):
            answer_text = ''.join(
                part.get('text', '') if isinstance(part, dict) else str(part)
                for part in content
            )
        else:
            answer_text = str(content)
    else:
        answer_text = str(response)

    output = {
        'answer': answer_text,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [30]:
# Example usage:
result = rag_advanced("Complex Curves in O-Minimal Geomatry", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Complex Curves in O-Minimal Geomatry'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the score threshold of 0.1.


TypeError: string indices must be integers, not 'str'